# Imaginaries of Artificial Intelligence



In this assignment you will map the imaginaries of a topic related to Artificial Intelligence on YouTube. 

This is a broad research goal, so you can adjust it to your own curiosity and preferences. 

The reason the assignment focuses YouTube specifically is due to the relative accessibility of its API. If you prefer so (/dare to), you can work with other data sources (e.g., another social media platform, search engine results, news/academic articles) for which you would have to develop a web scraper.


***Main steps***

* Obtain an API Key to be able to use the YouTube Data API (see: https://developers.google.com/youtube/v3/quickstart/python). Familiarize yourself with the API logic and, optionally, with the Python wrapper library youtube-data-api (see:https://pypi.org/project/youtube-data-api/).


* Agree on your research topic  / question. Example questions include: What are the main topics associated to AI? What are the main imaginaries around how AI will transform the world? What is the sentiment associated to different AI technologies?


* Define one or more queries that matche your topic / question. The query can be as broad as e.g., "Artificial Intelligence" or "Large Language Models", or something more specific e.g., "how LLM will change the world" or "the next AI development". It is good practice to test your queries on YouTube to make sure that they produce meaningful results.


* Decide what type of data to look at. Your entry point can be videos or channels, and for each you can decide to focus on different type of (textual) information (e.g., titles, descriptions, captions, comments, or a combination). You can also decide to apply a more refined data collection logic (e.g., you can look at a specific time frame, or a specific geographical area, etc.). In order to make inform decisions, consult the API documentaion to know what are the possibilities (see: https://developers.google.com/youtube/v3/docs/?apix=true)


* Collect the data for your research. When designing your data collection strategy, beware of requests' quotas and rate limits (https://developers.google.com/youtube/v3/determine_quota_cost).


* Develop and implement a data analysis strategy based on a NLP technique of your choice (or a cobination), that allow to extract insights from your corpus (see the suggested pointers below).

* Write a research report that includes: research question; logic of data collection; strategy for data analysis; main findings.


Ideally, you should come to the lab with the data already collected, and with an idea on how to proceed for the analysis.



 





***Pointers for data analysis***

*Topic Modelling: https://towardsdatascience.com/end-to-end-topic-modeling-in-python-latent-dirichlet-allocation-lda-35ce4ed6b3e0

*Word2Vec: https://towardsdatascience.com/a-beginners-guide-to-word-embedding-with-gensim-word2vec-model-5970fa56cc92

*KMeans: https://www.analyticsvidhya.com/blog/2021/04/k-means-clustering-simplified-in-python/

*Sentiment analysis: https://www.analyticsvidhya.com/blog/2022/07/sentiment-analysis-using-python/

We have generally not tried these particular implementations and they are just examples of what you could do. You are free to choose any analysis technique that allows you to derive meaningful insights from your corpus.

In [ ]:

import os
import re
import json
import time
from pathlib import Path
from datetime import datetime, timezone


import pandas as pd
import numpy as np


import matplotlib.pyplot as plt
import seaborn as sns


from tqdm import tqdm


from dotenv import load_dotenv

from googleapiclient.discovery import build


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer


pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 50)
sns.set(style="whitegrid")


In [ ]:

BASE_DIR = Path("..")  
CODE_DATA_DIR = BASE_DIR / "code_data"
DATA_DIR = CODE_DATA_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
CONFIG_DIR = CODE_DATA_DIR / "config"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

print("RAW_DIR:", RAW_DIR.resolve())
print("PROCESSED_DIR:", PROCESSED_DIR.resolve())
print("CONFIG_DIR:", CONFIG_DIR.resolve())

load_dotenv(CONFIG_DIR / ".env")

YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY")
if not YOUTUBE_API_KEY:
    raise ValueError(
        "YOUTUBE_API_KEY not found. Add it to code_data/config/.env (do NOT commit it)."
    )

youtube = build("youtube", "v3", developerKey=YOUTUBE_API_KEY)
print("YouTube API client initialized.")


RAW_DIR: C:\Users\jldee\Documents\UvA_AI-in-Society_2025-2026\code_data\data\raw
PROCESSED_DIR: C:\Users\jldee\Documents\UvA_AI-in-Society_2025-2026\code_data\data\processed
CONFIG_DIR: C:\Users\jldee\Documents\UvA_AI-in-Society_2025-2026\code_data\config
YouTube API client initialized.


In [ ]:

QUERIES_BASE = [
    "AI future",
    "future of artificial intelligence",
    "AI will change the world",
    "AI in 10 years",
]

QUERIES_UTOPIAN = [
    "AI revolution future",
    "AI will improve society",
    "AI benefits future",
    "AI breakthrough future",
    "AI optimism future",
]

QUERIES_DYSTOPIAN = [
    "AI danger future",
    "AI will replace jobs",
    "AI surveillance future",
    "AI existential risk",
    "AI takes over",
]

QUERY_GROUPS = {
    "base": QUERIES_BASE,
    "utopian": QUERIES_UTOPIAN,
    "dystopian": QUERIES_DYSTOPIAN,
}


MAX_RESULTS_PER_QUERY = 120   
RELEVANCE_LANGUAGE = "en"     
REGION_CODE = None           


published_after = datetime.now(timezone.utc).replace(microsecond=0)
published_after = published_after.replace(year=published_after.year - 1)
PUBLISHED_AFTER = published_after.isoformat().replace("+00:00", "Z")

print("Published after:", PUBLISHED_AFTER)
print("Total queries:", sum(len(v) for v in QUERY_GROUPS.values()))


def yt_search_video_ids(query, max_results=50, published_after=None, region_code=None, relevance_language=None):
    """
    Collect video IDs for a query using search.list (paginated).
    Note: search.list costs a lot of quota; keep max_results reasonable.
    """
    video_ids = []
    next_page = None

    while len(video_ids) < max_results:
        batch = min(50, max_results - len(video_ids))
        req = youtube.search().list(
            part="id",
            q=query,
            type="video",
            maxResults=batch,
            pageToken=next_page,
            publishedAfter=published_after,
            regionCode=region_code,
            relevanceLanguage=relevance_language,
            order="relevance",
        )
        res = req.execute()

        for item in res.get("items", []):
            vid = item["id"].get("videoId")
            if vid:
                video_ids.append(vid)

        next_page = res.get("nextPageToken")
        if not next_page:
            break

    return video_ids

all_hits = []

for group_name, query_list in QUERY_GROUPS.items():
    for q in query_list:
        vids = yt_search_video_ids(
            query=q,
            max_results=MAX_RESULTS_PER_QUERY,
            published_after=PUBLISHED_AFTER,
            region_code=REGION_CODE,
            relevance_language=RELEVANCE_LANGUAGE,
        )

        all_hits.extend([{
            "group": group_name,
            "query": q,
            "video_id": vid
        } for vid in vids])

hits_df = pd.DataFrame(all_hits).drop_duplicates()

print("Rows (query-video pairs):", len(hits_df))
print("Unique videos:", hits_df["video_id"].nunique())
hits_df.head()

hits_path = RAW_DIR / "search_hits.csv"
hits_df.to_csv(hits_path, index=False)
print("Saved:", hits_path)
hits_path = RAW_DIR / "search_hits.csv"
hits_df.to_csv(hits_path, index=False)
print("Saved:", hits_path)


Published after: 2025-01-29T21:39:38Z
Total queries: 14
Rows (query-video pairs): 1334
Unique videos: 882
Saved: ..\code_data\data\raw\search_hits.csv
Saved: ..\code_data\data\raw\search_hits.csv


In [ ]:
from googleapiclient.errors import HttpError

def yt_videos_details(video_ids):
    """
    Fetch video metadata in chunks of 50 via videos.list.
    Returns a dataframe with snippet + statistics + contentDetails.
    """
    rows = []

    for i in tqdm(range(0, len(video_ids), 50), desc="Fetching video metadata"):
        chunk = video_ids[i:i+50]
        try:
            req = youtube.videos().list(
                part="snippet,statistics,contentDetails",
                id=",".join(chunk)
            )
            res = req.execute()
        except HttpError as e:
            print("HttpError on chunk starting at", i, ":", e)
            continue

        for item in res.get("items", []):
            sn = item.get("snippet", {})
            st = item.get("statistics", {})
            cd = item.get("contentDetails", {})

            rows.append({
                "video_id": item.get("id"),
                "title": sn.get("title"),
                "description": sn.get("description"),
                "publishedAt": sn.get("publishedAt"),
                "channelId": sn.get("channelId"),
                "channelTitle": sn.get("channelTitle"),
                "categoryId": sn.get("categoryId"),
                "tags": json.dumps(sn.get("tags", []), ensure_ascii=False),
                "duration": cd.get("duration"),

                "viewCount": int(st.get("viewCount", 0)) if "viewCount" in st else 0,
                "likeCount": int(st.get("likeCount", 0)) if "likeCount" in st else np.nan,
                "commentCount": int(st.get("commentCount", 0)) if "commentCount" in st else np.nan,
            })

    return pd.DataFrame(rows)

video_ids = hits_df["video_id"].dropna().unique().tolist()
print("Unique video IDs to fetch:", len(video_ids))

videos_df = yt_videos_details(video_ids)

videos_labeled_df = videos_df.merge(hits_df, on="video_id", how="left")

print("videos_df shape:", videos_df.shape)
print("videos_labeled_df shape:", videos_labeled_df.shape)
videos_labeled_df.head()
videos_path = RAW_DIR / "videos.csv"
videos_labeled_df.to_csv(videos_path, index=False)
print("Saved:", videos_path)


def clean_text(s):
    if pd.isna(s):
        return ""
    s = re.sub(r"\s+", " ", str(s)).strip()
    return s

creator_df = videos_labeled_df.copy()
creator_df["title_clean"] = creator_df["title"].map(clean_text)
creator_df["desc_clean"] = creator_df["description"].map(clean_text)

creator_df["doc_creator"] = (creator_df["title_clean"] + " " + creator_df["desc_clean"]).str.strip()

print("Empty creator docs:", (creator_df["doc_creator"].str.len() == 0).sum())
creator_df[["video_id", "group", "query", "title"]].head()


Unique video IDs to fetch: 882


Fetching video metadata: 100%|██████████| 18/18 [00:02<00:00,  6.39it/s]


videos_df shape: (882, 12)
videos_labeled_df shape: (1334, 14)
Saved: ..\code_data\data\raw\videos.csv
Empty creator docs: 0


,video_id,group,query,title
0,1UufaK3pQMg,base,AI future,AI2027: Is this how AI might destroy humanity? - BBC World Service
1,1UufaK3pQMg,base,future of artificial intelligence,AI2027: Is this how AI might destroy humanity? - BBC World Service
2,1UufaK3pQMg,base,AI will change the world,AI2027: Is this how AI might destroy humanity? - BBC World Service
3,1UufaK3pQMg,base,AI in 10 years,AI2027: Is this how AI might destroy humanity? - BBC World Service
4,1UufaK3pQMg,utopian,AI revolution future,AI2027: Is this how AI might destroy humanity? - BBC World Service
